# 章节实践

通过本章的系统学习，我们已经掌握了使用Ascend C开发矢量算子的完整方法。为了巩固所学知识，现提供以下实践练习：

实现Ascend C算子Sigmoid,算子命名为SigmoidCustom，编写其kernel侧代码、host侧代码，并完成kernel直调测试。
相关算法：

                           sigmoid(x) = 1/(1 + exp(-x))

要求：

- 完成Sigmoid算子kernel侧核函数实现相关代码补齐。

- 完成Sigmoid算子host侧核函数调用函数相关代码补齐。

- 要支持Float类型输入输出。

请开始你的实践，体验从理解到创造的完整开发过程。

---

环境准备：

In [3]:
import os
!mkdir -p Sources
!bash -l -c 'source ~/.bashrc && env' > /tmp/shell_env.txt  # bash
with open("/tmp/shell_env.txt", "r") as f:
    for line in f.readlines():
        line = line.strip()
        if "=" in line and not line.startswith(("#", " ")):
            key, value = line.split("=", 1)
            os.environ[key] = value

---

sigmoid_custom.asc代码补齐：

In [ ]:
%%writefile Sources/sigmoid_custom.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <cmath> 
#include <algorithm>
#include <iterator>
#include "acl/acl.h"
#include "kernel_operator.h"

constexpr uint32_t BUFFER_NUM = 2; // tensor num for each queue

struct SigmoidCustomTilingData
{
    uint32_t totalLength;
    uint32_t tileNum;
};

// 待补充……

__global__ __aicore__ void sigmoid_custom(GM_ADDR x, GM_ADDR y, SigmoidCustomTilingData tiling)
{
    // 待补充……
}

std::vector<float> kernel_sigmoid(std::vector<float> &x)
{
    // 待补充……
}

uint32_t VerifyResult(std::vector<float> &output, std::vector<float> &golden)
{
    auto printTensor = [](std::vector<float> &tensor, const char *name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    printTensor(output, "Output");
    printTensor(golden, "Golden");
    if (std::equal(golden.begin(), golden.end(), output.begin())) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Case accuracy is verification failed!" << std::endl;
        return 1;
    }
    return 0;
}

std::vector<float> sigmoid(const std::vector<float>& x) {
    std::vector<float> y(x.size());
    for (size_t i = 0; i < x.size(); ++i) {
        y[i] = 1.0f / (1.0f + std::exp(-x[i]));
    }
    return y;
}

int32_t main(int32_t argc, char *argv[])
{
    constexpr uint32_t totalLength = 8 * 2048;
    constexpr float valueX = 5.5f;
    std::vector<float> x(totalLength, valueX);
    std::vector<float> output = kernel_sigmoid(x);
    std::vector<float> golden = sigmoid(x);
    return VerifyResult(output, golden);
}

---

生成CMakeLists.txt文件（直接执行，无需修改）

In [ ]:
%%writefile Sources/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)
find_package(ASC REQUIRED)
project(kernel_samples LANGUAGES ASC CXX)

add_executable(sigmoid_test
    sigmoid_custom.asc
)
target_compile_options(sigmoid_test PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=dav-2201>
)
target_link_libraries(sigmoid_test m)

---

执行以下命令进行编译并验证结果：

In [ ]:
!cd Sources && mkdir -p build && cd build
!cd Sources/build/ && cmake .. && make
!cd Sources/build/ && ./sigmoid_test

预期执行效果如下：

<img src="./images/执行结果05.png" alt="执行结果05"  width="800px" >

---

执行以下代码获取答案。

In [ ]:
!cat ./answer/02.05_answer.txt